In [22]:
import os
from pathlib import Path
import subprocess

repo_path = Path("/content/loan-default-mlbd")

if not repo_path.exists():
    !git clone https://github.com/kratipatel/loan-default-mlbd.git

%cd /content/loan-default-mlbd

!python -m src.prepare_data

/content/loan-default-mlbd
reading: https://raw.githubusercontent.com/tiwvis/MLBD-Projects/main/data_sets/LoansData_sample.csv.gz
saved: /content/loan-default-mlbd/data/processed/loans_clean.parquet (86141, 71)
        size      mean
split                 
test   12922  0.187587
train  60298  0.187585
val    12921  0.187602


In [23]:
import time
import json
import numpy as np
import tensorflow as tf

from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers

from src.data import load_splits, make_nn_data
from src.metrics import evaluate

tf.keras.utils.set_random_seed(42)

print("All libraries imported successfully!")

All libraries imported successfully!


In [18]:
from src.data import load_splits, make_nn_data

path = "data/processed/loans_clean.parquet"

tr, va, te = load_splits(path)

print("Train:", tr.shape)
print("Validation:", va.shape)
print("Test:", te.shape)

data, num_cols, cat_cols, cards = make_nn_data(tr, va, te)

print("\nNumeric features:", len(num_cols))
print("Categorical features:", len(cat_cols))
print("Category cardinalities:", cards)
print("Train arrays:", [x.shape for x in data["train"]])

Train: (60298, 71)
Validation: (12921, 71)
Test: (12922, 71)

Numeric features: 59
Categorical features: 9
Category cardinalities: [8, 36, 5, 4, 14, 47, 3, 2, 2]
Train arrays: [(60298, 59), (60298, 9), (60298,)]


In [19]:

# Set random seed for reproducibility
tf.keras.utils.set_random_seed(42)

# Numerical input (59 features)
num_input = keras.Input(shape=(len(num_cols),), name="numeric")

# Categorical inputs and embedding layers
cat_inputs = []
embeddings = []

for i, card in enumerate(cards):
    inp = keras.Input(
        shape=(1,),
        name=f"cat_{i}",
        dtype="int32"
    )

    emb_dim = min(16, max(2, int(card ** 0.5)))

    emb = layers.Embedding(
        input_dim=card,
        output_dim=emb_dim
    )(inp)

    emb = layers.Flatten()(emb)

    cat_inputs.append(inp)
    embeddings.append(emb)

# Combine numerical and categorical features
x = layers.Concatenate()([num_input] + embeddings)

# Dense layers
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)

# Binary classification output
output = layers.Dense(1, activation="sigmoid")(x)

# Create Keras model
model = keras.Model(
    inputs=[num_input] + cat_inputs,
    outputs=output
)

# Compile model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

# Display model architecture
model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cat_0 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_1 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_2 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_3 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_4 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_5 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_6 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_7 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_8 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_36        │ (None, 1, 2)      │         16 │ cat_0[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_37        │ (None, 1, 6)      │        216 │ cat_1[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_38        │ (None, 1, 2)      │         10 │ cat_2[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_39        │ (None, 1, 2)      │          8 │ cat_3[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_40        │ (None, 1, 3)      │         42 │ cat_4[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_41        │ (None, 1, 6)      │        282 │ cat_5[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_42        │ (None, 1, 2)      │          6 │ cat_6[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_43        │ (None, 1, 2)      │          4 │ cat_7[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_44        │ (None, 1, 2)      │          4 │ cat_8[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric             │ (None, 59)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_36          │ (None, 2)         │          0 │ embedding_36[0][

 Total params: 20,045 (78.30 KB)

 Trainable params: 20,045 (78.30 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
import time

Xn_train, Xc_train, y_train = data["train"]
Xn_val, Xc_val, y_val = data["val"]

def model_inputs(Xn, Xc):
    return [Xn] + [Xc[:, i:i+1] for i in range(Xc.shape[1])]

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

start_time = time.time()

history = model.fit(
    model_inputs(Xn_train, Xc_train),
    y_train,
    validation_data=(
        model_inputs(Xn_val, Xc_val),
        y_val
    ),
    epochs=30,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

keras_training_time = time.time() - start_time

print("Training time:", round(keras_training_time, 2), "seconds")

Epoch 1/30
236/236 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8029 - auc: 0.6776 - loss: 0.4638 - val_accuracy: 0.8147 - val_auc: 0.7266 - val_loss: 0.4346
Epoch 2/30
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8148 - auc: 0.7185 - loss: 0.4367 - val_accuracy: 0.8167 - val_auc: 0.7308 - val_loss: 0.4315
Epoch 3/30
236/236 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8162 - auc: 0.7267 - loss: 0.4320 - val_accuracy: 0.8174 - val_auc: 0.7338 - val_loss: 0.4296
Epoch 4/30
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8179 - auc: 0.7304 - loss: 0.4298 - val_accuracy: 0.8182 - val_auc: 0.7349 - val_loss: 0.4284
Epoch 5/30
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8180 - auc: 0.7348 - loss: 0.4274 - val_accuracy: 0.8163 - val_auc: 0.7352 - val_loss: 0.4280
Epoch 6/30
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8188 - auc: 0.7372 - loss: 0.4259 - val_accuracy: 0.8166 - val_auc: 0.7357 - val_loss: 0.4276
Epoch 7/30
236/236 ━━━━━━━━━━━━━━━━━━━━ 

In [21]:

# Predict probabilities
Xn_val, Xc_val, y_val = data["val"]
Xn_test, Xc_test, y_test = data["test"]

val_prob = model.predict(model_inputs(Xn_val, Xc_val), verbose=0).ravel()
test_prob = model.predict(model_inputs(Xn_test, Xc_test), verbose=0).ravel()

# Calculate metrics
keras_results = {
    "validation": evaluate(y_val, val_prob),
    "test": evaluate(y_test, test_prob),
    "training_time_seconds": round(keras_training_time, 2)
}

print(json.dumps(keras_results, indent=4))

# Save results
Path("reports").mkdir(exist_ok=True)
with open("reports/results_keras.json", "w") as f:
    json.dump(keras_results, f, indent=4)

print("\nSaved: reports/results_keras.json")

{
    "validation": {
        "rmse": 0.36704212996220614,
        "mae": 0.26908288451604023,
        "r2": 0.11605290439673088,
        "roc_auc": 0.7367700295322474,
        "pr_auc": 0.3970508468744704,
        "ks": 0.33870049622853116
    },
    "test": {
        "rmse": 0.3685611227262228,
        "mae": 0.27192102529121237,
        "r2": 0.10866833790917152,
        "roc_auc": 0.7329797652798238,
        "pr_auc": 0.38978311881188166,
        "ks": 0.3459906240195367
    },
    "training_time_seconds": 25.1
}

Saved: reports/results_keras.json
